# Adaptive Z-bias sampling for the frequency map

At each proposed Z bias, acquire a full drive-IF spectrum, estimate its resonance frequency, and update one GP model of f(Z). The next Z is chosen from the uncertainty of this map. A spectral peak is a candidate and needs independent validation.

Edit the JSON plan and review the exact total job and shot budget before running hardware. No candidate is published automatically.


In [ ]:
from pathlib import Path
import sys
ROOT = Path.cwd().resolve()
if ROOT.name == 'execute':
    ROOT = ROOT.parent
if not (ROOT / 'src').is_dir():
    raise FileNotFoundError('Start Jupyter in Platform_LQ or execute')
sys.path.insert(0, str(ROOT))
from agent_cli import read_spec, preflight, run
SPEC = ROOT / 'config/plans/zfreq_map_bo_example.json'
RUN_HARDWARE = True
spec = read_spec(SPEC)
preflight(spec)


In [ ]:
if RUN_HARDWARE:
    from src.progress import watch_run
    with watch_run(ROOT / "result" / "zfreq", preflight(spec)['jobs'],
                   run_mode=spec.get("run_mode", "new"), resume_run_tag=spec.get("resume_run_tag", ""),
                   title='Z 频率贝叶斯优化'):
        outcome = run(spec)
        print(outcome)
else:
    print('Dry plan only; no LQCloud submission.')


## Inspect saved frequency maps

Set PLOT_RUN_TAG to a completed or partial run directory to replay the Z-sampling sequence, its GP frequency map, and the measured spectra without submitting QPU jobs. Figures are saved in that run's fig/ directory.


In [ ]:
from optimize.show_fig import plot_bo_run

PLOT_RUN_TAG = ""  # For offline replay, paste one existing run directory name here.
run_dir = None
if RUN_HARDWARE:
    run_dir = Path(outcome["run_dir"])
elif PLOT_RUN_TAG:
    if (Path(PLOT_RUN_TAG).name != PLOT_RUN_TAG
            or "/" in PLOT_RUN_TAG or '\\' in PLOT_RUN_TAG):
        raise ValueError("PLOT_RUN_TAG must be one result directory name")
    run_dir = ROOT / "result" / 'zfreq' / PLOT_RUN_TAG

if run_dir is None:
    print("No data: set PLOT_RUN_TAG to a saved run or enable RUN_HARDWARE.")
else:
    figures = plot_bo_run(run_dir)
